# 03 · Modelo (MLflow + Registro)

**Configuración base:**
- Catálogo: `main`
- Schemas: `loterias_raw`, `loterias_bronze`, `loterias_silver`, `loterias_features`
- Volume crudos: `/Volumes/main/loterias_raw/raw_apuestas/apuestas_partitioned/`

In [0]:
CATALOG = "main"
TABLE_FEATURE = f"{CATALOG}.loterias_features.features_apuestas"
MLFLOW_EXPERIMENT = "/Shared/loti-mlops"
REGISTERED_NAME   = "loti_fraude_logit"

import mlflow, mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# 1) apunte de Registry a UC
mlflow.set_registry_uri("databricks-uc")

# 2) experimento (ajusta si usas otra ruta)
mlflow.set_experiment("/Shared/loti-mlops")

TABLE_FEATURE = "main.loterias_features.features_apuestas"
REGISTERED_NAME = "main.loterias_models.loti_fraude_logit"  # <== nombre UC

pdf = spark.table(TABLE_FEATURE).toPandas()
features = ["monto_log","freq_usuario","urgencia","repeticion_ip",
            "prom_monto_usuario","apuestas_7d_usuario","tasa_riesgo_ip"]
X, y = pdf[features], pdf["es_fraude"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

with mlflow.start_run(run_name="logit_baseline_v1"):
    model = LogisticRegression(max_iter=2000)
    model.fit(Xtr, ytr)
    y_prob = model.predict_proba(Xte)[:,1]
    mlflow.log_metric("roc_auc", float(roc_auc_score(yte, y_prob)))
    mlflow.log_params({"features": ",".join(features)})
    mlflow.sklearn.log_model(model, artifact_path="model")  # sin registered_model_name
